# Model Retention Walk-Forward Gate

Purpose: audit replayable SOXL models against the current retention target: complete walk-forward evidence, strict max drawdown below 20%, and YTD return aligned to both the converted SOXL backtest reference and current best replay model.

In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'AGENTS.md').exists():
    ROOT = next(parent for parent in Path.cwd().parents if (parent / 'AGENTS.md').exists())
CARD = ROOT / 'runtime/model_selection/card/model_retention_walkforward_gate_v1'
CARD.mkdir(parents=True, exist_ok=True)
ROOT, CARD

(PosixPath('/Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix'),
 PosixPath('/Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix/runtime/model_selection/card/model_retention_walkforward_gate_v1'))

In [2]:
import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.app.model_retention_gate import scan_model_retention_gates, write_retention_gate_outputs

result = scan_model_retention_gates(root=ROOT, max_walkforward_drawdown_pct=20.0, best_return_ratio=0.95)
write_retention_gate_outputs(result, output_dir=CARD)
summary = result['summary']
summary

{'generated_at': '2026-06-14T04:22:16.944362+00:00',
 'profiles_scanned': 0,
 'passing_profiles': [],
 'passing_profile_count': 0,
 'registry_keep_actor_count': 0,
 'registry_keep_actor_strict_violation_count': 0,
 'registry_keep_actor_disable_count': 0,
 'registry_keep_actor_protected_exception_count': 0,
 'registry_keep_actor_strict_violations': [],
 'reference_profile': 'soxl_backtest_policy_reimplementation_v1',
 'reference_return_pct': 132.077978,
 'reference_max_drawdown_pct': 13.554089,
 'current_best_return_pct': 224.361539,
 'best_return_ratio': 0.95,
 'required_best_return_pct': 213.143462,
 'max_walkforward_drawdown_pct': 20.0,
 'required_wf_windows': ['wf_2020', 'wf_2022', 'wf_2023_2024']}

In [3]:
rows = pd.DataFrame(result['rows'])
display(rows[['profile', 'status', 'ytd_return_pct', 'ytd_max_drawdown_pct', 'wf_windows_present', 'worst_wf_max_drawdown_pct', 'gate_pass', 'retention_decision', 'failures']])

,profile,status,ytd_return_pct,ytd_max_drawdown_pct,wf_windows_present,worst_wf_max_drawdown_pct,gate_pass,retention_decision,failures
0,soxl_backtest_policy_reimplementation_v1,reference,132.077978,13.554089,,None,True,keep_reference_baseline,[]


## Conclusion

- `soxl_backtest_policy_reimplementation_v1` is retained as the converted rule-policy reference baseline.
- No replayable RL/model profile currently satisfies all gates: complete `wf_2020`, `wf_2022`, `wf_2023_2024`; every WF MaxDD `<20%`; YTD return `>=132.077978%`; and YTD return `>=95%` of the current best replay (`smoke`, `224.361539%`).
- Strict final retention has been applied: every failed `keep_actor=true` registry row was downgraded to summary-only (`keep_actor=false`, `keep_summary=true`, `wf_status=blocked`); no actor files were deleted.
- After replay refresh, `dashboard_data/model_replay_freshness.json` has 0 retained executable qlib actor profiles in this checkout. Dashboard v2 therefore renders the SOXLBacktest reference baseline plus retained evidence exports, and only the deterministic SOXLBacktest baseline can rerun selected windows.
- The next model-training loop must produce a candidate that passes this card before it becomes a final retained executable model.